# Tutorial 1: When do large errors dominate a forecast?

## Briefing for the responsible study group

| Planning item | Allocation |
|---|---|
| Presentation date | Friday 20 November 2026 |
| Prerequisite lecture | Forecasting & Prediction, 10 November 2026 |
| Preparation time after prerequisite | 7 full weekdays, excluding lecture and tutorial dates |
| Format | 15-minute presentation + approximately 15-minute discussion |

**Lecture connection:** Forecasting slides 14–16 and 24–27: loss minimization, residuals, MAE and RMSE.

This is your group's working notebook. It replaces the requirement to augment a section of an existing course notebook. The supplied cells establish a starting point; your group develops and teaches the investigation.

### Your teaching challenge

The lecture introduces squared and absolute errors mathematically. Your task is to make their consequences tangible: how can a small number of extreme target values change a fitted regression model, and why can MAE and RMSE tell different stories about its predictions?

Investigate a simple, approximately linear resale-price relationship. Compare ordinary least squares with **one** robust alternative, such as `HuberRegressor`. Use the same training cases and the same held-out cases for both methods. Introduce extreme prices into the training targets only, keeping features, the underlying relationship and test data fixed. The supplied generator separates these ingredients so you can make a controlled comparison.

### Core investigation

1. Fit both models to the clean training data and establish their baseline performance.
2. Repeat after contaminating a small fraction of training prices. Show how the fitted lines change, and compare held-out MAE and RMSE using identical test observations.
3. Compare a few contamination settings, changing either the frequency or the magnitude of unusual prices. Keep this small; a broad robustness benchmark is unnecessary.
4. Explain why squared loss gives large residuals disproportionate influence. Distinguish **the loss used to fit a model** from **the metric used to assess it**. Explain what your robust alternative does differently at an intuitive level.

### What classmates should gain

They should be able to predict which model or metric will be more sensitive to a large residual and understand why there is no context-free rule to discard unusual observations. The clean test set here represents ordinary resale cases. Discuss how your recommendation might change if very high prices were legitimate, important cases in the deployment population rather than data errors.

### Boundaries and support

Use the supplied one-feature data and standard estimators. No derivation of the Huber optimizer or hyperparameter search is required. Keep asymmetric business costs outside this tutorial. **Optional:** change the evaluation population to include legitimate extremes, while keeping the training comparison controlled.

## How to turn your investigation into a tutorial

Your group has **15 minutes to present**, followed by approximately **15 minutes of discussion** with the instructor and class. Your audience has attended the prerequisite lecture but has not done your investigation. Help them understand one important question through a focused, reproducible demonstration.

**Required contribution.** Complete the core investigation below, explain why the results occur, and discuss at least one limitation or contrasting condition. Select two or three main figures or tables that make your argument clear. A useful tutorial connects the question, experimental design, evidence and conclusion; a sequence of unexplained code cells is insufficient. You are not expected to build a production system, implement algorithms from scratch, collect external data or run an extensive model search.

**What to submit.** Submit your completed `.ipynb` notebook with explanations and reproducible code, plus any slides or other presentation material you use. Follow the instructor's announced submission deadline. Keep the supplied briefing, but edit, reorganize or replace the working sections as needed. Identify substantial changes to supplied helpers. Record package versions, seeds and relevant settings; restart the kernel and run all cells before submission. Clearly distinguish supplied material from your group's contribution.

**Presentation format is your choice.** You may present from the notebook, create slides, render to HTML, develop an interactive demonstration or combine these. Make the selected format work within 15 minutes and have a fallback for live demonstrations. Creativity earns credit when it improves understanding, engagement or the quality of the investigation. Technical spectacle should be avoided. A clear static plot can be excellent.

**AI assistance is allowed.** You may use AI to help develop code, explanations and visualizations. Your group remains responsible for correctness and must be able to explain the submitted work. Include a short account of the tools used, their main contributions, and how you checked those contributions. Verify at least one important result independently—for example, with a hand calculation, an analytic special case or a comparison with a trusted library function. A long prompt transcript is not required. Using AI is optional and is not itself rewarded.

**Assessment.** This component contributes **15% of the course grade**. Within the tutorial, the criteria are **conceptual understanding (40%)**, **added value for classmates (40%)**, and **quality of communication (20%)**. Added value comes from the investigation and how it develops understanding beyond the lecture and the supplied starter cells. Optional extensions are not required for an excellent grade. All groups should prioritize a sound core investigation over breadth.

**Suggested presentation rhythm:** about 3 minutes for the question and essential concepts, 8 minutes for the demonstration and evidence, and 4 minutes for interpretation and limitations. Prepare one question that invites the class to reason about your findings. Anticipate questions about experimental controls, assumptions and alternative explanations.

## Preliminaries: environment

The course environment provides all libraries that you need to execute the startet codes (see [ReadMe for intallation instructions](https://github.com/stefanlessmann/ESMT_IML)). You can install additional libraries if needed. 

Run the supplied cells first. They provide a starting point, **not a completed tutorial**. Code cells marked `YOUR GROUP'S WORK` are intentionally unfinished and can be replaced or expanded.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

SEED = 888
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True,
                     'grid.alpha': 0.2, 'font.size': 11})
print(f'Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}')

## Preliminaries: supplied data and helpers

The generator returns the same clean and contaminated training observations, a fixed test set, and a grid for plotting regression lines. `suspect` records which training targets were altered; it is diagnostic metadata, not an input feature. Prices are synthetic and measured in arbitrary monetary units. The generator is deliberately simple and does not enforce positive prices after extreme contamination.

In [ ]:
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

def make_resale_data(seed=SEED, contamination=0.05, magnitude=500.0):
    if not 0 <= contamination <= 1:
        raise ValueError('contamination must be between 0 and 1')
    rng = np.random.default_rng(seed)
    age = rng.uniform(6, 48, 500)
    price = 1000 - 16 * age + rng.normal(0, 45, len(age))
    train_idx, test_idx = np.arange(350), np.arange(350, 500)
    contaminated = price[train_idx].copy()
    # Stable random ordering: increasing contamination adds cases to the same set.
    order = rng.permutation(len(train_idx))
    suspect = np.zeros(len(train_idx), dtype=bool)
    suspect[order[:round(contamination * len(train_idx))]] = True
    contaminated[suspect] += magnitude
    return {
        'X_train': age[train_idx, None], 'y_train_clean': price[train_idx],
        'y_train_contaminated': contaminated, 'suspect': suspect,
        'X_test': age[test_idx, None], 'y_test': price[test_idx],
        'X_grid': np.linspace(6, 48, 200)[:, None],
    }

data = make_resale_data()
display(pd.DataFrame({'age': data['X_train'][:, 0],
                      'clean_price': data['y_train_clean'],
                      'observed_price': data['y_train_contaminated'],
                      'suspect': data['suspect']}).head())

## Investigation 1: Establish a fair baseline

Fit OLS and one robust alternative to the clean training prices. Define a reusable evaluation table with MAE and RMSE on the fixed test set. Use np.sqrt(mean_squared_error(...)) for RMSE.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 2: Introduce unusual training prices

Repeat with contaminated training targets. Plot the observations, indicate altered cases, and overlay both fitted lines. What changed, and what did you hold fixed?

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 3: Explain sensitivity and its limits

Vary contamination modestly. Include one hand calculation showing how one residual contributes to absolute and squared error. Explain when the clean test population is appropriate and when it is not.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Your teaching conclusions

Replace this text with:

1. A direct answer to the tutorial's central question, supported by your results.
2. Two or three takeaways that classmates should remember.
3. One limitation, counterexample or condition under which the result changes.
4. One discussion question, plus notes on how your group would answer it.

## Contribution and verification record

- **Supplied material retained or changed:** …
- **Your group's experimental and explanatory contributions:** …
- **Independent check of an important result:** …
- **AI tools and their contributions, if any:** …
- **How generated code/explanations were checked:** …
- **Package versions, random seeds and reproduction instructions:** …
- **Group members and contributions:** …

## Before presenting

- [ ] Restarted the kernel and ran all cells successfully.
- [ ] Checked that splits, metrics, units and class labels are correct.
- [ ] Distinguished evidence from interpretation; avoided claims stronger than the experiment supports.
- [ ] Labelled plots and made figures legible for classmates.
- [ ] Rehearsed within 15 minutes and prepared for discussion.
- [ ] Included the completed notebook and any additional presentation material.